# CTFJudge — Train and held-out test analysis

This notebook analyzes the 160-item train campaign, documents the prompt-selection decision, and evaluates the single frozen run on the 40-item held-out test set. It uses only Python's standard library plus Matplotlib.

In [1]:
from pathlib import Path
import json
from collections import Counter, defaultdict
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / 'experiments').exists():
    ROOT = ROOT.parent
TRACKING = ROOT / 'experiments' / 'prompt-tracking'
TEST_ROOT = ROOT / 'experiments' / 'test-evaluation' / 'final-v014'
LABELS = ['excellent', 'good', 'average', 'weak']
COLORS = ['#55d6be', '#4f9cf9', '#f7c948', '#ef6f6c']
plt.style.use('seaborn-v0_8-darkgrid')

In [2]:
def read_jsonl(path):
    return [json.loads(line) for line in path.read_text(encoding='utf-8').splitlines() if line.strip()]

train_versions = []
train_rows = []
for number in range(4, 21):
    version = f'v{number:03d}'
    folder = TRACKING / version
    metrics = json.loads((folder / 'metrics.json').read_text(encoding='utf-8'))
    rows = read_jsonl(folder / 'results.jsonl')
    train_versions.append({'version': version, **metrics, 'samples': len(rows)})
    train_rows.extend([{**row, 'version': version} for row in rows])

print('Train samples:', len(train_rows))
print('Train completed:', sum(row['status'] == 'COMPLETED' for row in train_rows))
assert len(train_rows) == 160

Train samples: 160
Train completed: 160


In [3]:
versions = [item['version'] for item in train_versions]
accuracies = [item['accuracy'] * 100 for item in train_versions]
samples = [item['samples'] for item in train_versions]
fig, ax = plt.subplots(figsize=(14, 5))
bars = ax.bar(versions, accuracies, color='#4f9cf9')
ax.set(title='Train exact-label accuracy by prompt version', ylabel='Accuracy (%)', xlabel='Prompt version', ylim=(0, 70))
for bar, value, n in zip(bars, accuracies, samples):
    ax.text(bar.get_x() + bar.get_width()/2, value + 1, f'{value:.0f}%\nn={n}', ha='center', fontsize=8)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

C:\Users\PC\AppData\Local\Temp\ipykernel_40236\1401315641.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [4]:
pairs = [('L02', 'v007', 'v008'), ('L03', 'v009', 'v010'), ('L04', 'v011', 'v012'), ('L05', 'v013', 'v014'), ('L06', 'v015', 'v016'), ('L07', 'v017', 'v018'), ('L08', 'v019', 'v020')]
pair_stats = []
for lab, high_v, low_v in pairs:
    rows = [row for row in train_rows if row['version'] in {high_v, low_v}]
    correct = sum(row.get('predicted') == row['expected'] for row in rows if row['status'] == 'COMPLETED')
    completed = sum(row['status'] == 'COMPLETED' for row in rows)
    pair_stats.append((lab, correct, completed, correct / completed if completed else 0))

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar([x[0] for x in pair_stats], [x[3]*100 for x in pair_stats], color='#55d6be')
ax.set(title='Paired train accuracy by lab (high + low batches)', ylabel='Accuracy (%)', ylim=(0, 60))
for idx, (_, correct, total, acc) in enumerate(pair_stats):
    ax.text(idx, acc*100 + 1, f'{correct}/{total}', ha='center')
plt.tight_layout()
plt.show()
pair_stats

C:\Users\PC\AppData\Local\Temp\ipykernel_40236\2652974487.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


[('L02', 8, 19, 0.42105263157894735),
 ('L03', 3, 19, 0.15789473684210525),
 ('L04', 4, 19, 0.21052631578947367),
 ('L05', 9, 19, 0.47368421052631576),
 ('L06', 5, 19, 0.2631578947368421),
 ('L07', 8, 20, 0.4),
 ('L08', 6, 20, 0.3)]

## Why prompt v014 was selected

Selection was completed before reading held-out test results. We compare paired high/low batches within each lab, because individual batches contain different label mixes. The L05 pair (`v013 + v014`) achieved **9/19 = 47.37%**, the strongest paired train result. `v014` itself achieved **5/9 = 55.56%** on average/weak reports, the main failure mode of earlier prompts.

The selected prompt preserves tables and evidence appendices, audits the complete report rather than isolated chunks, deduplicates report-level defects, and accepts internally supported run variants such as PID, timestamp, path, filename, privilege and environment outcome. It is frozen for the test run; the test set is not used for further prompt tuning.

In [5]:
analysis_dir = ROOT / 'experiments' / 'analysis'
train_metric_path = analysis_dir / 'train-runs.jsonl'
metric_train = read_jsonl(train_metric_path) if train_metric_path.exists() else []
if not metric_train:
    print('Run scripts/cache_analysis_metrics.py to populate token and latency charts.')
else:
    by_version = {}
    for version in versions:
        rows = [row for row in metric_train if row['version'] == version]
        by_version[version] = {
            'input': sum(row['input_tokens'] for row in rows),
            'output': sum(row['output_tokens'] for row in rows),
            'latency': sum(row['latency_seconds'] for row in rows) / len(rows),
            'speed': sum(row['inference_tokens_per_minute'] for row in rows) / len(rows),
        }
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    axes[0].bar(versions, [by_version[v]['input'] for v in versions], label='Input tokens', color='#4f9cf9')
    axes[0].bar(versions, [by_version[v]['output'] for v in versions], bottom=[by_version[v]['input'] for v in versions], label='Output tokens', color='#55d6be')
    axes[0].set(title='Train token cost by version', ylabel='Tokens'); axes[0].legend()
    axes[1].bar(versions, [by_version[v]['latency'] for v in versions], color='#f7c948'); axes[1].set(title='Mean train latency', ylabel='Seconds / submission')
    axes[2].bar(versions, [by_version[v]['speed'] for v in versions], color='#ef6f6c'); axes[2].set(title='Mean inference speed', ylabel='Output tokens / minute')
    for ax in axes: ax.tick_params(axis='x', rotation=60)
    plt.tight_layout(); plt.show()

C:\Users\PC\AppData\Local\Temp\ipykernel_40236\454668567.py:23: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


In [6]:
def confusion(rows):
    matrix = [[0 for _ in LABELS] for _ in LABELS]
    for row in rows:
        if row['status'] == 'COMPLETED':
            matrix[LABELS.index(row['expected'])][LABELS.index(row['predicted'])] += 1
    return matrix

def plot_confusion(matrix, title):
    fig, ax = plt.subplots(figsize=(6, 5))
    image = ax.imshow(matrix, cmap='Blues')
    ax.set_xticks(range(4), LABELS, rotation=30, ha='right')
    ax.set_yticks(range(4), LABELS)
    ax.set(xlabel='Predicted', ylabel='Expected', title=title)
    for i in range(4):
        for j in range(4):
            ax.text(j, i, matrix[i][j], ha='center', va='center')
    fig.colorbar(image, ax=ax, fraction=0.046)
    plt.tight_layout()
    plt.show()

selected_train = [row for row in train_rows if row['version'] in {'v013', 'v014'}]
plot_confusion(confusion(selected_train), 'Selected prompt family: train confusion matrix')

C:\Users\PC\AppData\Local\Temp\ipykernel_40236\2217944064.py:19: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [7]:
test_result_path = TEST_ROOT / 'results.jsonl'
if not test_result_path.exists():
    print('Held-out test has not finished yet. Run scripts/run_test_evaluation.py first.')
    test_rows = []
else:
    test_rows = read_jsonl(test_result_path)
    completed = [row for row in test_rows if row['status'] == 'COMPLETED']
    accuracy = sum(row['expected'] == row['predicted'] for row in completed) / len(completed)
    print(f'Test completed: {len(completed)}/40')
    print(f'Test exact-label accuracy: {accuracy:.2%}')

Test completed: 40/40
Test exact-label accuracy: 30.00%


In [8]:
if test_rows:
    plot_confusion(confusion(test_rows), 'Held-out test confusion matrix — frozen v014 prompt')
    labs = sorted({row['lab_id'] for row in test_rows})
    lab_accuracy = []
    for lab in labs:
        rows = [row for row in test_rows if row['lab_id'] == lab and row['status'] == 'COMPLETED']
        lab_accuracy.append(sum(row['expected'] == row['predicted'] for row in rows) / len(rows))
    fig, ax = plt.subplots(figsize=(12, 4))
    ax.bar([lab.split('_', 1)[0] for lab in labs], [x*100 for x in lab_accuracy], color='#f7c948')
    ax.set(title='Held-out test accuracy by lab', ylabel='Accuracy (%)', ylim=(0, 100))
    plt.tight_layout()
    plt.show()

C:\Users\PC\AppData\Local\Temp\ipykernel_40236\2217944064.py:19: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


C:\Users\PC\AppData\Local\Temp\ipykernel_40236\3375708010.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [9]:
if test_rows:
    matrix = confusion(test_rows)
    print('Per-label precision / recall / F1')
    for idx, label in enumerate(LABELS):
        tp = matrix[idx][idx]
        fp = sum(matrix[row][idx] for row in range(4)) - tp
        fn = sum(matrix[idx]) - tp
        precision = tp / (tp + fp) if tp + fp else 0
        recall = tp / (tp + fn) if tp + fn else 0
        f1 = 2*precision*recall/(precision+recall) if precision+recall else 0
        print(f'{label:10s} precision={precision:.3f} recall={recall:.3f} f1={f1:.3f}')

Per-label precision / recall / F1
excellent  precision=0.000 recall=0.000 f1=0.000
good       precision=0.280 recall=0.500 f1=0.359
average    precision=0.500 recall=0.417 f1=0.455
weak       precision=0.000 recall=0.000 f1=0.000


In [10]:
test_metric_path = analysis_dir / 'test-runs.jsonl'
metric_test = read_jsonl(test_metric_path) if test_metric_path.exists() else []
if metric_test:
    train_input = sum(row['input_tokens'] for row in metric_train)
    train_output = sum(row['output_tokens'] for row in metric_train)
    test_input = sum(row['input_tokens'] for row in metric_test)
    test_output = sum(row['output_tokens'] for row in metric_test)
    train_latency = sum(row['latency_seconds'] for row in metric_train) / len(metric_train)
    test_latency = sum(row['latency_seconds'] for row in metric_test) / len(metric_test)
    train_speed = sum(row['inference_tokens_per_minute'] for row in metric_train) / len(metric_train)
    test_speed = sum(row['inference_tokens_per_minute'] for row in metric_test) / len(metric_test)
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    axes[0].bar(['Train', 'Test'], [train_input, test_input], label='Input', color='#4f9cf9')
    axes[0].bar(['Train', 'Test'], [train_output, test_output], bottom=[train_input, test_input], label='Output', color='#55d6be')
    axes[0].set(title='Total token cost', ylabel='Tokens'); axes[0].legend()
    axes[1].bar(['Train', 'Test'], [train_latency, test_latency], color='#f7c948'); axes[1].set(title='Mean latency', ylabel='Seconds / submission')
    axes[2].bar(['Train', 'Test'], [train_speed, test_speed], color='#ef6f6c'); axes[2].set(title='Mean inference speed', ylabel='Output tokens / minute')
    plt.tight_layout(); plt.show()
    print({'train_tokens': train_input + train_output, 'test_tokens': test_input + test_output, 'train_mean_latency_s': round(train_latency, 2), 'test_mean_latency_s': round(test_latency, 2), 'train_tokens_per_min': round(train_speed, 2), 'test_tokens_per_min': round(test_speed, 2)})

C:\Users\PC\AppData\Local\Temp\ipykernel_40236\853498549.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


{'train_tokens': 5292292, 'test_tokens': 1428581, 'train_mean_latency_s': 167.75, 'test_mean_latency_s': 182.62, 'train_tokens_per_min': 4908.39, 'test_tokens_per_min': 4930.01}


## Interpretation checklist

- Compare train-pair accuracy with held-out accuracy to identify overfitting.
- Inspect the confusion matrix for systematic over-scoring or under-scoring.
- Compare labs to identify behavior types that need better golden chunking or retrieval.
- Treat this test run as final evaluation. Any later prompt change requires a new validation protocol, not another tuning pass on these 40 labels.